<a href="https://colab.research.google.com/github/zeyneptass/vector-databases-and-rag-fundamentals/blob/main/SemanticSearchDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pinecone sentence-transformers tqdm

In [ ]:
from google.colab import userdata

In [ ]:
from pinecone import Pinecone, ServerlessSpec

In [ ]:
# Secret Manager'a kaydettiğiniz isimle çağırın
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

# Kontrol amaçlı (anahtarın ilk 5 karakterini gösterir)
print(f"API Key yüklendi: {PINECONE_API_KEY[:5]}...")

# Pinecone istemcisini oluşturun (pc değişkeni burada tanımlanıyor)
pc = Pinecone(api_key=PINECONE_API_KEY)

API Key yüklendi: pcsk_...


In [ ]:
index_name = "semantic-search-demo"

# Aynı isimde eski bir dizin varsa sil (temiz başlangıç için)
if index_name in pc.list_indexes().names():
    print(f"'{index_name}' dizini zaten var, siliniyor...")
    pc.delete_index(index_name)
    print("Silindi.")
else:
    print(f"'{index_name}' dizini mevcut değil, oluşturulacak.")

'semantic-search-demo' dizini mevcut değil, oluşturulacak.


In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding modeli yüklendi.")
print(f"Model çıktı boyutu: {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding modeli yüklendi.
Model çıktı boyutu: 384


/tmp/ipykernel_7245/3179314567.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model çıktı boyutu: {model.get_sentence_embedding_dimension()}")


In [ ]:
dimension = model.get_sentence_embedding_dimension()  # 384

pc.create_index(
    name=index_name,
    dimension=dimension,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)
print(f"'{index_name}' dizini oluşturuldu.")

/tmp/ipykernel_7245/2583887717.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = model.get_sentence_embedding_dimension()  # 384


'semantic-search-demo' dizini oluşturuldu.


In [ ]:
# Dizin hazır olana kadar bekle
while not pc.describe_index(index_name).status.ready:
    print("Dizin hazırlanıyor...")
    time.sleep(2)

print("Dizin kullanıma hazır!")

Dizin kullanıma hazır!


In [ ]:
index = pc.Index(index_name)
print(index)

Index(host='https://semantic-search-demo-h0jgoc3.svc.aped-4627-b74a.pinecone.io')


In [ ]:
# Dış bağlantıya ihtiyaç duymayan, kendi soru listemiz
questions = [
    "Which city has the highest population in the world?",
    "What is the most beautiful city in the world?",
    "Which country has the fastest-growing population?",
    "How do I make a chocolate cake?",
    "How do I make a delicious cake?",
    "How do you make a 10-inch cake?",
    "What is the capital of France?",
    "What is the largest country by area?",
    "How does photosynthesis work?",
    "What is the meaning of life?",
    "How do I learn Python programming?",
    "What is the best way to learn machine learning?",
    "Which programming language should I learn first?",
    "What is the difference between AI and machine learning?",
    "How do neural networks work?",
    "What is a vector database?",
    "How does semantic search work?",
    "What is Pinecone used for?",
    "How do I install Python packages?",
    "What is the weather like today?"
] * 50  # 20 soruyu 50 kez tekrarlayarak 1000 soru elde ediyoruz

print(f"Toplam {len(questions)} soru hazırlandı.")
print("İlk 5 soru:")
for q in questions[:5]:
    print(f"  - {q}")

Toplam 1000 soru hazırlandı.
İlk 5 soru:
  - Which city has the highest population in the world?
  - What is the most beautiful city in the world?
  - Which country has the fastest-growing population?
  - How do I make a chocolate cake?
  - How do I make a delicious cake?


In [ ]:
from tqdm.auto import tqdm

batch_size = 100  # Her seferde 100 vektör gönder

for i in tqdm(range(0, len(questions), batch_size)):
    # Bu partideki soruları al
    batch = questions[i:i + batch_size]

    # Soruları vektörlere çevir
    embeddings = model.encode(batch).tolist()

    # Pinecone'un beklediği formatta hazırla: (id, vektör, metadata)
    vectors_to_upsert = []
    for j, (question, embedding) in enumerate(zip(batch, embeddings)):
        vector_id = f"q_{i + j}"  # Benzersiz kimlik
        metadata = {"text": question}  # Orijinal metni metadata'ya koy
        vectors_to_upsert.append((vector_id, embedding, metadata))

    # Pinecone'a yükle
    index.upsert(vectors=vectors_to_upsert)

print("Tüm sorular Pinecone'a yüklendi!")

  0%|          | 0/10 [00:00<?, ?it/s]

Tüm sorular Pinecone'a yüklendi!


In [ ]:
stats = index.describe_index_stats()
print(stats)
print(f"Toplam vektör sayısı: {stats['total_vector_count']}")

DescribeIndexStatsResponse(dimension=384, total_vector_count=1000, metric='cosine', namespaces=1)
Toplam vektör sayısı: 1000


In [ ]:
def run_query(query_text, top_k=5):
    """
    Bir sorgu metnini alır, vektöre çevirir ve Pinecone'da arar.
    """
    # Sorguyu vektöre çevir
    query_embedding = model.encode(query_text).tolist()

    # Pinecone'da ara
    results = index.query(
        vector=query_embedding,
        top_k=top_k,              # En benzer kaç sonuç gelsin
        include_metadata=True     # Metadata'yı da getir
    )

    # Sonuçları yazdır
    print(f"\nSorgu: '{query_text}'")
    print("-" * 50)
    for match in results['matches']:
        score = match['score']
        text = match['metadata']['text']
        print(f"Benzerlik: {score:.4f} | Soru: {text}")
    print("-" * 50)

    return results

In [ ]:
# Test 1: Şehirlerle ilgili soru
run_query("Which city has the highest population in the world?")

# Test 2: Kek tarifiyle ilgili soru
run_query("How do I make a chocolate cake?")

# Test 3: Programlama ile ilgili soru
run_query("How do I learn Python?")


Sorgu: 'Which city has the highest population in the world?'
--------------------------------------------------
Benzerlik: 1.0017 | Soru: Which city has the highest population in the world?
Benzerlik: 1.0017 | Soru: Which city has the highest population in the world?
Benzerlik: 1.0017 | Soru: Which city has the highest population in the world?
Benzerlik: 1.0017 | Soru: Which city has the highest population in the world?
Benzerlik: 1.0017 | Soru: Which city has the highest population in the world?
--------------------------------------------------

Sorgu: 'How do I make a chocolate cake?'
--------------------------------------------------
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
--------------------------------------------------

Sorgu: 'How

QueryResponse(matches=[ScoredVector(id='q_150', score=0.961209357, values=[], metadata={'text': 'How do I learn Python programming?'}), ScoredVector(id='q_170', score=0.961209357, values=[], metadata={'text': 'How do I learn Python programming?'}), ScoredVector(id='q_190', score=0.961209357, values=[], metadata={'text': 'How do I learn Python programming?'}), ScoredVector(id='q_10', score=0.961209357, values=[], metadata={'text': 'How do I learn Python programming?'}), ScoredVector(id='q_530', score=0.961209357, values=[], metadata={'text': 'How do I learn Python programming?'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Thu, 08 Oct 2026 01:31:08 GMT', 'content-type': 'application/json', 'content-length': '602', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '10', 'x-pinecone-request-latency-ms': '52', 'x-envoy-upstream-service-time': '53', 'x-pinecone-response-duration-ms': '54', 'grpc-status': '0', 'server'

In [ ]:
run_query("Which city has the highest population?")
run_query("How do I make a chocolate cake?")
run_query("What is the latest technology news?")


Sorgu: 'Which city has the highest population?'
--------------------------------------------------
Benzerlik: 0.9674 | Soru: Which city has the highest population in the world?
Benzerlik: 0.9674 | Soru: Which city has the highest population in the world?
Benzerlik: 0.9674 | Soru: Which city has the highest population in the world?
Benzerlik: 0.9674 | Soru: Which city has the highest population in the world?
Benzerlik: 0.9674 | Soru: Which city has the highest population in the world?
--------------------------------------------------

Sorgu: 'How do I make a chocolate cake?'
--------------------------------------------------
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
Benzerlik: 1.0004 | Soru: How do I make a chocolate cake?
--------------------------------------------------

Sorgu: 'What is the late

QueryResponse(matches=[ScoredVector(id='q_339', score=0.235916167, values=[], metadata={'text': 'What is the weather like today?'}), ScoredVector(id='q_179', score=0.235916167, values=[], metadata={'text': 'What is the weather like today?'}), ScoredVector(id='q_359', score=0.235916167, values=[], metadata={'text': 'What is the weather like today?'}), ScoredVector(id='q_379', score=0.235916167, values=[], metadata={'text': 'What is the weather like today?'}), ScoredVector(id='q_939', score=0.235916167, values=[], metadata={'text': 'What is the weather like today?'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Thu, 08 Oct 2026 01:33:44 GMT', 'content-type': 'application/json', 'content-length': '588', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '10', 'x-pinecone-request-latency-ms': '47', 'x-envoy-upstream-service-time': '48', 'x-pinecone-response-duration-ms': '49', 'grpc-status': '0', 'server': 'envoy'}))